# Replaying Experimental Recordings in Flybody

In this tutoriel we replay a snippet of experimentally recorderd fly walking on a ball in the (flybody model)[https://github.com/TuragaLab/flybody/tree/dev] from the Turaga lab. 
The data is the same as the one presented in the original NeuroMechFly publication. Custom code based on seqikpy was used to extract the kinematic chain and run the inverse kinematics.

## Load the motion snippet

The data has been bundeled into a clip at `src/flygym_demo/ball_flybody_data/assets/ball_flybody_clip.npz`. It already matches the `MotionSnippet` schema, so we can load it directly and move on to replaying the motion.

In [1]:
from pathlib import Path
import numpy as np
from tqdm import trange

from flygym.compose import KinematicPosePreset, ActuatorType, FlatGroundWorld
from flygym.assets.model.flybody.anatomy_flybody import (
    FlybodySkeleton,
    FlybodyJointPreset,
    FlybodyAxisOrder,
    FlybodyActuatedDOFPreset,
    FlybodyContactBodiesPreset,
)
from flygym.utils.math import Rotation3D
from flygym.compose.fly import FlybodyFly
from flygym import Simulation
from flygym_demo.spotlight_data import MotionSnippet

snippet_path = Path("/Users/stimpfli/Desktop/flygym-v2/src/flygym_demo/ball_flybody_data/assets/ball_flybody_clip.npz")
snippet = MotionSnippet(snippet_path, angles_global2anatomical=False)

print("Experimental data and metadata:")
print(f"  legs: {snippet.legs}")
print(f"  dofs_per_leg: {snippet.dofs_per_leg}")
print(f"  data_fps: {snippet.data_fps}")
print(f"  experiment_trial: {snippet.experiment_trial}")
print(f"  framerange_in_raw_recording: {snippet.framerange_in_raw_recording}")
print(f"  joint_angles: shape={snippet.joint_angles.shape}, dtype={snippet.joint_angles.dtype}")
print(f"  fwdkin_egoxyz: shape={snippet.fwdkin_egoxyz.shape}, dtype={snippet.fwdkin_egoxyz.dtype}")
print(f"  rawpred_egoxyz: shape={snippet.rawpred_egoxyz.shape}, dtype={snippet.rawpred_egoxyz.dtype}")

Experimental data and metadata:
  legs: ['lf', 'lm', 'lh', 'rf', 'rm', 'rh']
  dofs_per_leg: [('thorax', 'coxa', 'pitch'), ('thorax', 'coxa', 'roll'), ('thorax', 'coxa', 'yaw'), ('coxa', 'trochanterfemur', 'pitch'), ('coxa', 'trochanterfemur', 'roll'), ('trochanterfemur', 'tibia', 'pitch'), ('tibia', 'tarsus1', 'pitch')]
  data_fps: 100.0
  experiment_trial: generated_from_ik_results
  framerange_in_raw_recording: [0, 100]
  joint_angles: shape=(100, 6, 7), dtype=float32
  fwdkin_egoxyz: shape=(100, 0, 3), dtype=float32
  rawpred_egoxyz: shape=(100, 0, 3), dtype=float32


## Build the Flybody model

The model is configured with leg-only articulation, position actuators, tendon actuators, and leg adhesion. This keeps the notebook close to the original replay tutorial while exposing the Flybody-specific tendon capabilities.

In [2]:
axis_order = FlybodyAxisOrder.YAW_ROLL_PITCH
articulated_joints = FlybodyJointPreset.LEGS_ONLY
actuated_dofs = FlybodyActuatedDOFPreset.LEGS_ACTIVE_ONLY
actuator_type = ActuatorType.POSITION
neutral_pose = KinematicPosePreset.FLYBODY_NEUTRAL

fly = FlybodyFly()
skeleton = FlybodySkeleton(axis_order=axis_order, joint_preset=articulated_joints)
fly.add_joints(skeleton, neutral_pose)

position_actuated_dofs = fly.skeleton.get_actuated_dofs_from_preset(actuated_dofs)
fly.add_actuators(
    position_actuated_dofs,
    actuator_type=actuator_type,
)
fly.add_tendons()
fly.add_tendon_actuators()
fly.add_leg_adhesion()
fly.colorize()
tracking_cam = fly.add_tracking_camera()

bodysegs_with_ground_contact = FlybodyContactBodiesPreset.LEGS_THORAX_ABDOMEN_HEAD
spawn_position = (0, 0, 2.0)
spawn_rotation = Rotation3D("quat", (1, 0, 0, 0))

world = FlatGroundWorld()
world.add_fly(
    fly,
    spawn_position,
    spawn_rotation,
    bodysegs_with_ground_contact=bodysegs_with_ground_contact,
    add_ground_contact_sensors=True,
)

sim = Simulation(world)
sim.set_renderer(tracking_cam)

print("Flybody model ready")
print("  position actuators:", len(position_actuated_dofs))
print("  tendon actuators:", len(fly.get_actuated_jointdofs_order(ActuatorType.TENDON)))

Applying wing default pose correction.
Flybody model ready
  position actuators: 42
  tendon actuators: 6


## Replay the recording

The target joint trajectories are obtained through `MotionSnippet.get_joint_angles`, which handles smoothing, interpolation, and DOF reordering for the simulator. During the replay itself we keep the tendon actuators neutral.

In [3]:
sim_timestep = sim.mj_model.opt.timestep
position_targets = snippet.get_joint_angles(
    output_timestep=sim_timestep,
    output_dof_order=fly.get_actuated_jointdofs_order(actuator_type),
    sgfilter_window_sec=0.05
)

In [4]:


nsteps_sim = position_targets.shape[0]
n_dofs = len(fly.get_jointdofs_order())
n_position_actuators = len(fly.get_actuated_jointdofs_order(actuator_type))
n_tendon_actuators = len(fly.get_actuated_jointdofs_order(ActuatorType.TENDON))

simulated_joint_angles = np.full((nsteps_sim, n_dofs), np.nan, dtype=np.float32)
position_actuator_forces = np.full((nsteps_sim, n_position_actuators), np.nan, dtype=np.float32)

fly_name = fly.name
zero_tendon_inputs = np.zeros(n_tendon_actuators, dtype=np.float32)

sim.reset()
sim.set_tendon_actuator_inputs(fly_name, zero_tendon_inputs)

for step_idx in trange(nsteps_sim, desc="Replaying"):
    target_angles = position_targets[step_idx, :]
    sim.set_actuator_inputs(fly_name, actuator_type, target_angles)
    sim.step_with_profile()
    simulated_joint_angles[step_idx, :] = sim.get_joint_angles(fly_name)
    position_actuator_forces[step_idx, :] = sim.get_actuator_forces(fly_name, actuator_type)
    sim.render_as_needed_with_profile()

print("Replay complete")
print("  replay steps:", nsteps_sim)
print("  actuator target shape:", position_targets.shape)

Replaying: 100%|██████████| 10000/10000 [00:02<00:00, 4779.97it/s]

Replay complete
  replay steps: 10000
  actuator target shape: (10000, 42)


## Show the replay

After the replay finishes, we can display the rendered video inline and inspect the runtime profile.

In [5]:
sim.print_performance_report()
sim.renderer.show_in_notebook()

PERFORMANCE PROFILE


Stage,Time/step (us),Percent (%),Throughput (iters/s),Throughput x realtime
Physics simulation advancement,109,54,9180,0.92
Rendering*,93,46,10778,1.08
TOTAL,202,100,4957,0.50


* Note: 124 frames were rendered out of 10000 steps. Therefore, rendering time per image is 7483 us.


nmf/trackcam This browser does not support the video tag.


## Tendon actuation demo

Interesting the flybody model incorporated tendons to more accurately actuate some body parts. Lets create a fly with abdomen joints and use the tendon to actuate the abdomen. 

In [6]:
from flygym.compose import TetheredWorld

fly = FlybodyFly()
all_biological_joints = FlybodyJointPreset.ALL_BIOLOGICAL
all_biological_actuators = FlybodyActuatedDOFPreset.ALL
skeleton = FlybodySkeleton(axis_order=axis_order, joint_preset=all_biological_joints)
fly.add_joints(skeleton, neutral_pose)

actuated_dofs = fly.skeleton.get_actuated_dofs_from_preset(all_biological_actuators)
fly.add_actuators(
    actuated_dofs,
    actuator_type=actuator_type,
    kp=1.0, # just to silence the warning
)

fly.add_tendons()
fly.add_tendon_actuators()

fly.colorize()
tracking_cam = fly.add_tracking_camera(pos_offset=[-1, -5, 1], rotation=Rotation3D("xyaxes", (0.999, 0.044, -0.000, -0.012, 0.265, 0.964)))

world = TetheredWorld()
world.add_fly(fly,
              (0, 0, 0.0),
              Rotation3D("quat", (1, 0, 0, 0)))

sim = Simulation(world)
sim.set_renderer(tracking_cam)

Applying wing default pose correction.


In [7]:
nsteps_tendon = int(2.0 / sim_timestep)
hold_position = np.zeros(len(actuated_dofs), dtype=np.float32)
n_tendon_actuators = len(fly.get_actuated_jointdofs_order(ActuatorType.TENDON))
tendon_drive = np.zeros((nsteps_tendon, n_tendon_actuators), dtype=np.float32)
if tendon_drive.shape[1] > 0:
    tendon_drive[:, 0] = 0.35 * np.sin(np.linspace(0, 8 * np.pi, nsteps_tendon))

sim.reset()

for step_idx in trange(nsteps_tendon, desc="Tendon demo"):
    sim.set_actuator_inputs(fly_name, actuator_type, hold_position)
    sim.set_tendon_actuator_inputs(fly_name, tendon_drive[step_idx])
    sim.step_with_profile()
    sim.render_as_needed_with_profile()

sim.renderer.show_in_notebook()

Tendon demo: 100%|██████████| 20000/20000 [00:02<00:00, 7350.73it/s]


nmf/trackcam This browser does not support the video tag.
